# Lesson 4: The first neural network

Goes with the Lesson 4 page of the Forkcast course. The deliveries stop following a straight line; you watch the linear model fail, then build an MLP that bends.

Runtime: CPU is enough.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. The new deliveries, and the old model

`make_orders(nonlinear=True)`: rain slows every kilometre, and past 7 km riders take the bike-lane detour.

In [ ]:
import torch
from torch import nn
from forkcast.data import make_orders, FEATURES
from forkcast.model import LinearRegressor
from forkcast.preprocess import Normalizer
from forkcast import train

X, y = make_orders(2000, seed=0, nonlinear=True)
norm = Normalizer().fit(X)
Xn = norm.transform(X)

torch.manual_seed(0)
lin = LinearRegressor(4)
lin_losses = train.fit(lin, Xn, y, epochs=300, lr=0.1)
print(f"linear model: loss {lin_losses[-1]:.2f}   (noise floor is about 4)")

### The residual plot

Prediction minus actual, against distance, coloured by rain. A model that captured everything leaves a shapeless band around zero.

In [ ]:
import matplotlib.pyplot as plt

def residual_plot(model, title):
    with torch.no_grad():
        residual = model(Xn) - y
    plt.figure(figsize=(7, 3.5))
    plt.scatter(X[:, 0], residual, s=6, c=X[:, 2], cmap="coolwarm")
    plt.axhline(0, color="k", ls="--"); plt.xlabel("km"); plt.ylabel("prediction - actual"); plt.title(title)

residual_plot(lin, "LinearRegressor: the bend at 7 km and the rain drift are both visible")

## 3. Write `count_parameters` and `MLP`

In the cell below, under `LinearRegressor`. Build a list of layers in a loop over `hidden`, append the output layer, wrap with `nn.Sequential(*layers)`.

In [ ]:
%%writefile forkcast/model.py
"""Lesson 3. The delivery-time model as an nn.Module.

Replace every ``raise NotImplementedError`` with one to three lines of code.
"""

from __future__ import annotations

import torch
from torch import Tensor, nn


class LinearRegressor(nn.Module):
    """minutes = X @ w + b, with the knobs owned by an nn.Linear layer."""

    def __init__(self, n_features: int):
        super().__init__()
        raise NotImplementedError("Lesson 3, step 3: self.linear = nn.Linear(n_features, 1)")

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,). nn.Linear returns (N, 1); squeeze the last dimension away."""
        raise NotImplementedError("Lesson 3, step 3: return self.linear(X).squeeze(-1)")

# ---- Lesson 4: the first neural network -------------------------------------

def count_parameters(model: nn.Module) -> int:
    """Number of trainable numbers in ``model`` (sum of numel over parameters that require grad)."""
    raise NotImplementedError("Lesson 4, step 4: sum(p.numel() for p in model.parameters() if p.requires_grad)")


class MLP(nn.Module):
    """A multilayer perceptron: Linear -> activation -> ... -> Linear(…, 1).

    ``hidden`` lists the width of each hidden layer, so MLP(4, (32, 32)) is
    4 -> 32 -> 32 -> 1 with an activation after each hidden layer and none
    after the last. ``activation`` is a class such as nn.ReLU; MLP(4, ())
    has no hidden layers and is exactly LinearRegressor.
    """

    def __init__(self, n_features: int, hidden: tuple[int, ...] = (32, 32), activation=nn.ReLU):
        super().__init__()
        raise NotImplementedError("Lesson 4, step 4: build a list of layers, wrap it in nn.Sequential as self.net")

    def forward(self, X: Tensor) -> Tensor:
        """(N, d) -> (N,)"""
        raise NotImplementedError("Lesson 4, step 4: return self.net(X).squeeze(-1)")

In [ ]:
import importlib
from forkcast import model as model_module
importlib.reload(model_module)
from forkcast.model import MLP, count_parameters

model = MLP(4, (32, 32))
print(model.net)
print(model(Xn).shape)                      # torch.Size([2000])
print(count_parameters(model))              # 1249
print(count_parameters(MLP(4, ())))         # 5: no hidden layers is a straight line

## 4. Train it with the Lesson 3 toolkit, unchanged

In [ ]:
torch.manual_seed(0)
model = MLP(4, (32, 32))
losses = train.fit(model, Xn, y, epochs=300, lr=0.01, optimizer_cls=torch.optim.Adam)
print(f"MLP: loss {losses[0]:.1f} -> {losses[-1]:.2f}")
plt.plot(lin_losses, label="linear"); plt.plot(losses, label="MLP(4, (32, 32))")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend();
residual_plot(model, "MLP: a band around zero, no trend")

### Capacity

Same recipe, different widths. Training loss only; Lesson 5 adds the loss on unseen deliveries.

In [ ]:
for hidden in [(8,), (32,), (32, 32), (64, 64)]:
    torch.manual_seed(0)
    m = MLP(4, hidden)
    l = train.fit(m, Xn, y, epochs=300, lr=0.01, optimizer_cls=torch.optim.Adam)
    print(f"MLP(4, {hidden!s:10s}) {count_parameters(m):5d} parameters   loss {l[-1]:.2f}")

### Break it on purpose, twice

No activation: 1249 parameters, the loss of a straight line. SGD at Lesson 3's learning rate: nan.

In [ ]:
torch.manual_seed(0)
flat = train.fit(MLP(4, (32, 32), activation=nn.Identity), Xn, y, epochs=300, lr=0.01, optimizer_cls=torch.optim.Adam)
print("no activation:", round(flat[-1], 2), "  (linear model was", round(lin_losses[-1], 2), ")")

torch.manual_seed(0)
sgd = train.fit(MLP(4, (32, 32)), Xn, y, epochs=300, lr=0.1)
print("SGD lr=0.1:", sgd[-1])

## 5. Definition of done

Expected: `7 passed`, then `34 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_04.py -q

In [ ]:
!pytest -m fast -q

## 6. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 4: the first neural network"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main